# 2. Model Training: Naive MLP Baseline & InfoNCE Model
This notebook loads the cached visual and audio features, and trains two MLP heads to map (image, audio) $\rightarrow$ teacher video embedding.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np

# Load source code modules (we will clone the local repo inside Kaggle or load them directly)
import sys
sys.path.append('.')
from src.models import MLPApproximator
from src.loss import InfoNCELoss
from src.dataset import MultimodalEmbeddingDataset
from src.metrics import calculate_proximity, calculate_retrieval_metrics

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


## Step 1: Load Datasets


In [ ]:
train_dataset = MultimodalEmbeddingDataset(file_path="train_features.pt")
test_dataset = MultimodalEmbeddingDataset(file_path="test_features.pt")

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

print(f"Loaded {len(train_dataset)} training items, {len(test_dataset)} test items.")


## Step 2: Train Baseline 1 (MLP + Cosine Similarity Loss)
This serves as the baseline: concatenate image (512) and audio (128) embeddings and project them to teacher dimension (1024) using a simple MLP trained with cosine similarity loss.


In [ ]:
model_baseline = MLPApproximator(input_dim=640, hidden_dims=[512, 1024], output_dim=1024).to(device)
optimizer = optim.AdamW(model_baseline.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = lambda pred, target: (1 - torch.nn.functional.cosine_similarity(pred, target)).mean()

epochs = 30
for epoch in range(epochs):
    model_baseline.train()
    epoch_loss = 0.0
    for batch in train_loader:
        z_img = batch['z_img'].to(device)
        z_aud = batch['z_aud'].to(device)
        v_teacher = batch['v_teacher'].to(device)
        
        optimizer.zero_grad()
        v_pred = model_baseline(z_img, z_aud)
        loss = criterion(v_pred, v_teacher)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * z_img.size(0)
        
    train_loss = epoch_loss / len(train_dataset)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train Cosine Loss: {train_loss:.4f}")

# Save the model weights
torch.save(model_baseline.state_dict(), "mlp_cosine.pt")
print("Baseline 1 training complete and weights saved!")


## Step 3: Train Method A (MLP + InfoNCE Loss)
Instead of minimizing cosine distance independently, we optimize alignment over batches using symmetric InfoNCE contrastive loss.


In [ ]:
model_infonce = MLPApproximator(input_dim=640, hidden_dims=[512, 1024], output_dim=1024).to(device)
optimizer = optim.AdamW(model_infonce.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = InfoNCELoss(temperature=0.07, symmetric=True)

epochs = 30
for epoch in range(epochs):
    model_infonce.train()
    epoch_loss = 0.0
    for batch in train_loader:
        z_img = batch['z_img'].to(device)
        z_aud = batch['z_aud'].to(device)
        v_teacher = batch['v_teacher'].to(device)
        
        optimizer.zero_grad()
        v_pred = model_infonce(z_img, z_aud)
        loss = criterion(v_pred, v_teacher)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item() * z_img.size(0)
        
    train_loss = epoch_loss / len(train_dataset)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{epochs:02d} | Train InfoNCE Loss: {train_loss:.4f}")

# Save the model weights
torch.save(model_infonce.state_dict(), "mlp_infonce.pt")
print("Method A (InfoNCE) training complete and weights saved!")
